# Thousands of points without thousands of markers

DOM markers are excellent for a handful of important places. Large point collections belong in a GeoJSON source rendered by WebGL. This notebook generates 6,000 observations and lets MapLibre cluster them dynamically as the user zooms.

The dataset is deterministic and generated locally, so this is both a visual demonstration and a repeatable performance smoke test.

In [ ]:
from random import Random
from time import perf_counter

from maplibreum import Map

cities = {
    "Hiroshima": [132.4553, 34.3853],
    "Osaka": [135.5023, 34.6937],
    "Kyoto": [135.7681, 35.0116],
    "Kobe": [135.1955, 34.6901],
    "Okayama": [133.9344, 34.6551],
}
category_colors = {
    "crossing": "#e63946",
    "footway": "#457b9d",
    "kerb": "#2a9d8f",
}
rng = Random(42)
started = perf_counter()
features = []
for index in range(6_000):
    city = list(cities)[index % len(cities)]
    longitude, latitude = cities[city]
    category = list(category_colors)[index % len(category_colors)]
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [longitude + rng.gauss(0, 0.055), latitude + rng.gauss(0, 0.04)],
        },
        "properties": {
            "id": index + 1,
            "city": city,
            "category": category,
            "color": category_colors[category],
        },
    })
observations = {"type": "FeatureCollection", "features": features}
print(f"Generated {len(features):,} points in {perf_counter() - started:.3f} s")

## Cluster in the browser

`add_clustered_geojson` creates the source plus three coordinated layers: cluster circles, count labels, and individual points. Clusters expand and regroup automatically as the zoom changes.

In [ ]:
clustered_map = Map(
    center=[134.4, 34.75],
    zoom=6,
    title="Synthetic accessibility observations",
    height="620px",
)
clustered = clustered_map.add_clustered_geojson(
    observations,
    name="accessibility-observations",
    radius=55,
    max_zoom=13,
)
clustered_map.add_feature_popup(
    clustered.unclustered_layer_id,
    fields=["id", "city", "category"],
    aliases=["Observation", "City", "Feature type"],
    title="Mapped observation",
)
clustered_map.add_control("navigation", options={"showZoom": True})
clustered_map.add_control("fullscreen")

render_started = perf_counter()
rendered_html = clustered_map.render()
print(f"Rendered {len(rendered_html) / 1_000_000:.2f} MB of standalone HTML in {perf_counter() - render_started:.3f} s")

clustered_map

## Scale beyond embedded GeoJSON

Clustering improves browser rendering, but embedding millions of coordinates still creates an enormous HTML file. For truly large datasets, keep the same WebGL styling approach and move the data into vector tiles or PMTiles.

A useful rule of thumb:

- a few meaningful places → markers;
- thousands of points → a GeoJSON source with clustering;
- regional or global datasets → vector tiles or PMTiles.